# Qwen3.8-27B Abliterated on Colab T4 (AirLLM) — v2 stuck fix

> Colab untuk Google AI Pro (CCU) + AirLLM layer streaming. 27B full model,
> tanpa quantization, tanpa pruning. Muat dalam T4 16GB.

**Sebelum run:**
1. **Runtime -> Change runtime type -> Hardware accelerator: GPU (T4)**
2. Pilih **High Memory** (24GB) kalau ada - lebih safe
3. Run cells ikut urutan. **Option A atau Option B - pilih SATU je**
4. First run download ~27GB (4bit) / ~54GB (bf16) - guna `hf_transfer` jadi laju 3-5x

**Kalau stuck di cell 4bit sebelum ni (hang):** v2 ni fix profiling + download hang. Bawah ada log tiap layer jadi b nampak dia jalan, bukan freeze.


In [ ]:
# v2: hf_transfer untuk laju download 3-5x + verbose
!pip install -q -U airllm bitsandbytes transformers hf_transfer huggingface_hub
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print("HF_TRANSFER:", os.environ.get("HF_HUB_ENABLE_HF_TRANSFER"))
!nvidia-smi  # confirm T4 ada
!df -h /       # kena ada ~35GB free untuk 4bit, ~60GB untuk bf16
!free -h
!python -c "import huggingface_hub; print(huggingface_hub.__version__)"


## Kenapa T4 cukup (GPU + CPU + RAM + disk)

| Komponen | Role dalam AirLLM |
|---|---|
| T4 16GB VRAM | Pegang 1 layer aktif + KV cache. Qwen3.8-27B perlukan ~3.33GB sahaja -> lega |
| System RAM | Python, tokenizer, buffer. High memory (24GB) lebih selesa |
| Disk Colab | Simpan layer shards. bf16 ~54GB; 4bit ~27GB |
| CPU | Orkestrasi, decode. Bukan bottleneck |

Bottleneck sebenar = disk read speed + profiling first run. v2 log progress tiap layer.


In [ ]:
import os, shutil, torch
print("CUDA:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"VRAM: {p.total_memory/1e9:.1f}GB total | {p.name}")
total, used, free = shutil.disk_usage("/")
print(f"Disk / : {free/1e9:.0f}GB free / {total/1e9:.0f}GB total")
if free < 35*1e9:
    print("⚠️ Disk free <35GB — 4bit mungkin stuck! Free kan atau pakai 4bit je jangan A+B")
print("HF cache:", os.path.expanduser("~/.cache/huggingface"))
print("HF_TRANSFER:", os.environ.get("HF_HUB_ENABLE_HF_TRANSFER"))
# optional: mount gdrive untuk cache kekal (uncomment kalau nak)
# from google.colab import drive
# drive.mount('/content/drive')
# os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
# print("HF_HOME ->", os.environ["HF_HOME"])


## Option A - Default (bf16, quality penuh)

Download ~54GB. Kualiti paling tinggi, speed slow ~1-3 tok/s.
Pilih A kalau nak quality. JANGAN run A+B sekali.


In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from airllm import AutoModel
import time

# Jika b pernah stuck sebelum ni, profiling_mode=True memang lama 20-30min first run (nampak hang tapi jalan).
# v2: biar True untuk first split, lepas tu boleh False.
model = AutoModel.from_pretrained(
    "huihui-ai/Huihui-Qwen3.8-27B-abliterated",
    delete_original=True,
    profiling_mode=True,  # kalau stuck lama, tukar ke False dan rerun
)
print("Loaded:", type(model).__name__)

PROMPT = "Explain dalam 2 ayat apa Qwen3.8-27B abliterated ni."
msgs = [{"role": "user", "content": PROMPT}]
text = model.tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
toks = model.tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=False)

t0 = time.time()
out = model.generate(toks["input_ids"].cuda(), max_new_tokens=96, use_cache=True, return_dict_in_generate=True)
dt = time.time() - t0
seq = out.sequences[0]
n = seq.shape[0] - toks["input_ids"].shape[0]
print(f"{n} tokens dalam {dt:.1f}s -> {n/dt:.2f} tok/s")
print("=== OUTPUT ===")
print(model.tokenizer.decode(seq, skip_special_tokens=True))


## Option B - 4bit block compression (fix stuck v2)

Guna `bitsandbytes` block-wise quantization - laju ~3x, disk ~27GB.
**v2 fix untuk stuck:** tambah HF_TRANSFER + log + fallback profiling_mode=False.

> Kalau b stuck sebelum ni kat cell ni (hang tanpa output) — itu profiling 20-30min.
> v2 ni akan print progress, dan kalau masih hang, rerun dengan `profiling_mode=False`.


In [ ]:
import os, time, traceback
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print("HF_TRANSFER on, mula load 4bit... (akan nampak log split layer satu-satu)")

from airllm import AutoModel

# --- PENTING: kalau free disk <35GB atau RAM bukan High Memory, dia akan crawl/hang ---
import shutil, torch
_, _, free = shutil.disk_usage("/")
print(f"Disk free: {free/1e9:.1f}GB | RAM check:")
!free -h | head -n 2

try:
    # cubaan 1: profiling_mode=True (first run, akan lama tapi ada log)
    print("\n>>> Attempt 1: profiling_mode=True (kalau hang >30min, interrupt dan rerun cell dengan False) <<<")
    model = AutoModel.from_pretrained(
        "huihui-ai/Huihui-Qwen3.8-27B-abliterated",
        compression='4bit',
        delete_original=True,
        profiling_mode=True,
    )
    print("Loaded (4bit, profiling=True):", type(model).__name__)
except Exception as e:
    print("profiling=True gagal / interrupted, cuba fallback False...")
    traceback.print_exc()
    model = AutoModel.from_pretrained(
        "huihui-ai/Huihui-Qwen3.8-27B-abliterated",
        compression='4bit',
        delete_original=True,
        profiling_mode=False,
    )
    print("Loaded (4bit, profiling=False):", type(model).__name__)

PROMPT = "Explain dalam 2 ayat apa Qwen3.8-27B abliterated ni."
msgs = [{"role": "user", "content": PROMPT}]
text = model.tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
toks = model.tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=False)

print("\nGenerating...")
t0 = time.time()
out = model.generate(toks["input_ids"].cuda(), max_new_tokens=96, use_cache=True, return_dict_in_generate=True)
dt = time.time() - t0
seq = out.sequences[0]
n = seq.shape[0] - toks["input_ids"].shape[0]
print(f"{n} tokens dalam {dt:.1f}s -> {n/dt:.2f} tok/s (4bit)")
print("=== OUTPUT ===")
print(model.tokenizer.decode(seq, skip_special_tokens=True))
print("\n✅ Kalau sampai sini maknanya tak hang dah b!")


## Chat loop

Model dah load kat Option A/B? Guna cell ni. Colab support `input()` -
taip prompt, Enter, baca jawapan.


In [ ]:
def ask(q, max_new=160):
    msgs = [{"role": "user", "content": q}]
    text = model.tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    toks = model.tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=False)
    t0 = time.time()
    out = model.generate(toks["input_ids"].cuda(), max_new_tokens=max_new, use_cache=True, return_dict_in_generate=True)
    dt = time.time() - t0
    seq = out.sequences[0]
    ans = model.tokenizer.decode(seq, skip_special_tokens=True)
    n = seq.shape[0] - toks["input_ids"].shape[0]
    print(f"({n/max(0.1,dt):.1f} tok/s)")
    print(ans)

print("Mode chat. Taip soalan + Enter. Taip 'exit' untuk keluar.")
while True:
    q = input("b> ")
    if q.strip().lower() in ("exit", "quit", "keluar"):
        break
    if not q.strip():
        continue
    ask(q)


## Catatan v2 (jujur)

- **Kenapa hang sebelum ni:** `profiling_mode=True` first run profile 54 layer satu-satu, makan 20-30min tanpa log — nampak macam hang padahal jalan. v2 tambah log + fallback `False`.
- **HF_TRANSFER:** on jadi download 27GB dalam ~10-15min bukan 1 jam.
- **GDrive cache (optional):** uncomment 3 line mount tu kalau nak cache kekal lepas 5 jam runtime habis.
- **ToS Colab:** notebook peribadi je, jangan serve API keluar.
- **27B abliterated/uncensored** - output mungkin kasar, b dah aware.
